In [ ]:
# In[1]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Load files
metric = pd.read_csv("metric.csv")
trace = pd.read_csv("trace.csv")
log = pd.read_csv("log.csv")
error_logs = pd.read_csv("error_logs.csv")

# Helper to convert timestamps (seconds) to Asia/Shanghai timezone-aware datetimes
def to_shanghai_ts(s):
    # s can be a Series or scalar
    return pd.to_datetime(s, unit='s', utc=True).dt.tz_convert(tz)

# --- metric.csv summary ---
metric_rows = metric.shape[0]
metric_min_ts = to_shanghai_ts(metric['timestamp']).min()
metric_max_ts = to_shanghai_ts(metric['timestamp']).max()
metric_cmdb_list = pd.Series(sorted(metric['cmdb_id'].dropna().unique()))[:50]

# Top 20 kpi_name counts
metric_kpi_counts = metric.groupby('kpi_name').size().reset_index(name='count').sort_values('count', ascending=False).head(20)

# Aggregate by (cmdb_id, kpi_name) across full series: P95, P90, P50, P5, count
agg_funcs = {
    'P95': lambda x: x.quantile(0.95),
    'P90': lambda x: x.quantile(0.90),
    'P50': lambda x: x.quantile(0.50),
    'P5':  lambda x: x.quantile(0.05),
    'count': 'count'
}
metric_agg = metric.groupby(['cmdb_id', 'kpi_name'])['value'].agg(**agg_funcs).reset_index()
# sort by P95 desc and limit top 50
metric_agg_sorted = metric_agg.sort_values('P95', ascending=False).head(50)
# round numeric columns for compact display
metric_agg_sorted[['P95','P90','P50','P5']] = metric_agg_sorted[['P95','P90','P50','P5']].round(6)

# Compact summary dataframe for metric file
metric_summary = pd.DataFrame([{
    'file': 'metric.csv',
    'rows': metric_rows,
    'min_timestamp_shanghai': metric_min_ts,
    'max_timestamp_shanghai': metric_max_ts
}])

# --- trace.csv summary ---
trace_rows = trace.shape[0]
trace_min_ts = to_shanghai_ts(trace['timestamp']).min()
trace_max_ts = to_shanghai_ts(trace['timestamp']).max()
trace_cmdb_list = pd.Series(sorted(trace['cmdb_id'].dropna().unique()))[:50]
trace_name_counts = trace.groupby('trace_name').size().reset_index(name='count').sort_values('count', ascending=False).head(20)

trace_summary = pd.DataFrame([{
    'file': 'trace.csv',
    'rows': trace_rows,
    'min_timestamp_shanghai': trace_min_ts,
    'max_timestamp_shanghai': trace_max_ts
}])

# --- log.csv summary ---
log_rows = log.shape[0]
log_min_ts = to_shanghai_ts(log['timestamp']).min()
log_max_ts = to_shanghai_ts(log['timestamp']).max()
log_cmdb_list = pd.Series(sorted(log['cmdb_id'].dropna().unique()))[:50]
log_name_counts = log.groupby('log_name').size().reset_index(name='count').sort_values('count', ascending=False).head(20)

log_summary = pd.DataFrame([{
    'file': 'log.csv',
    'rows': log_rows,
    'min_timestamp_shanghai': log_min_ts,
    'max_timestamp_shanghai': log_max_ts
}])

# --- error_logs.csv summary ---
err_rows = error_logs.shape[0]
err_min_ts = to_shanghai_ts(error_logs['timestamp']).min()
err_max_ts = to_shanghai_ts(error_logs['timestamp']).max()
err_cmdb_list = pd.Series(sorted(error_logs['cmdb_id'].dropna().unique()))[:50]

error_summary = pd.DataFrame([{
    'file': 'error_logs.csv',
    'rows': err_rows,
    'min_timestamp_shanghai': err_min_ts,
    'max_timestamp_shanghai': err_max_ts
}])

# Final compact outputs (keep each piece concise). 
# Display order: metric summary + components, metric top agg, trace summary + top names, log summary + top names, error summary.
metric_summary, metric_cmdb_list, metric_kpi_counts, metric_agg_sorted, trace_summary, trace_cmdb_list, trace_name_counts, log_summary, log_cmdb_list, log_name_counts, error_summary, err_cmdb_list

```
Out[1]:
```
summary = (
    "Summary of telemetry files (compact):\n\n"
    "metric.csv:\n"
    "- Total rows: 27,436\n"
    "- Timestamp range (Asia/Shanghai, UTC+8): 2021-03-04 05:30:00+08:00 to 2021-03-04 06:00:00+08:00\n"
    "- Unique cmdb_id samples (28 total, showing some): IG01, IG02, MG01, MG02, Mysql01, Redis01, Redis02, Tomcat01, Tomcat02, apache01, apache02, dockerA1, dockerA2, dockerB1\n"
    "- Top KPI names by occurrence (top examples):\n"
    "  * OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies — 372\n"
    "  * OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT — 372\n"
    "  * OSLinux-OSLinux_ZABBIX_Host_Uptime — 371\n"
    "- Aggregation by (cmdb_id, kpi_name) computed globally (P95, P90, P50, P5, count). Top rows (by P95) include:\n"
    "  * Tomcat01 / Tomcat-Requests... — P95 ≈ 7.956886e+09, count 31\n"
    "  * Tomcat02 / Tomcat-Requests... — P95 ≈ 7.856477e+09, count 31\n"
    "  * Redis01 / network packets IN — P95 ≈ 4.858536e+09, count 15\n"
    "  * Redis01 / network packets OUT — P95 ≈ 4.385705e+09, count 15\n"
    "  * Redis02 / redis (maxmemory) — P95 = 4.294967e+09, count 30\n"
    "  (The aggregated table of up to 50 rows sorted by P95 was produced.)\n\n"
    "trace.csv:\n"
    "- Total rows: 4,155\n"
    "- Timestamp range (Asia/Shanghai, UTC+8): 2021-03-04 05:30:00+08:00 to 2021-03-04 05:59:00+08:00\n"
    "- Unique cmdb_id samples: IG01, IG02, MG01, MG02, Tomcat01..Tomcat04, dockerA1, dockerA2, dockerB1, dockerB2\n"
    "- Top trace names by count (top examples): trace.self.duration_p95 (302), trace.self.row_count (302), trace.self.duration_mean (302)\n\n"
    "log.csv:\n"
    "- Total rows: 386\n"
    "- Timestamp range (Asia/Shanghai, UTC+8): 2021-03-04 05:30:00+08:00 to 2021-03-04 05:59:00+08:00\n"
    "- Unique cmdb_id samples: Tomcat01, Tomcat02, Tomcat03, Tomcat04, apache01, apache02\n"
    "- Top log names by count (top examples): log.error_count (161), log.row_count (161), log.gc_pause_max (64)\n\n"
    "error_logs.csv:\n"
    "- Total rows: 0 (no timestamps or cmdb_id entries present)\n\n"
    "Notes:\n"
    "- All percentiles and aggregations for metric.csv were computed globally (over the full series) before any time filtering, as required.\n"
    "- The metric aggregation output (P95/P90/P50/P5/count) is available and was limited to the top 50 rows by P95 for compactness.\n"
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(         file   rows    min_timestamp_shanghai    max_timestamp_shanghai
0  metric.csv  27436 2021-03-04 05:30:00+08:00 2021-03-04 06:00:00+08:00, 0         IG01
1         IG02
2         MG01
3         MG02
4      Mysql01
        ...   
23    apache01
24    apache02
25    dockerA1
26    dockerA2
27    dockerB1
Length: 28, dtype: object,                                               kpi_name  count
266      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies    372
255     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT    372
278                 OSLinux-OSLinux_ZABBIX_Host_Uptime    371
271         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount    371
256       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT    371
165                            OSLinux-CPU_CPU_CPULoad    369
168                             OSLinux-CPU_CPU_CPUWio    369
167                        OSLinux-CPU_CPU_CPUUserTime    369
169                        OSLinux-CPU_CPU_CPUidleutil    369
166                         OSLinux-CPU_CPU_CPUSysTime    369
270  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...    367
254              OSLinux-OSLinux_MEMORY_MEMORY_UserMem    292
267          OSLinux-OSLinux_PROCESS_PROCESS_PROCPPMem    292
252       OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc    292
249             OSLinux-OSLinux_MEMORY_MEMORY_CacheMem    291
250           OSLinux-OSLinux_MEMORY_MEMORY_MEMFreeMem    291
253       OSLinux-OSLinux_MEMORY_MEMORY_NoCacheMemPerc    286
164                         OSLinux-CPU_CPU_CPUCpuUtil    286
221  OSLinux-OSLinux_FILE_-tmp-zabbix_agentd.log_Fi...    279
257    OSLinux-OSLinux_NETWORK_NETWORK_TotalTcpConnNum    223,        cmdb_id                                           kpi_name           P95           P90           P50            P5  count
1172  Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_Processin...  7.956886e+09  7.956884e+09  7.956878e+09  7.956865e+09     31
1277  Tomcat02  Tomcat-Requests_7441-"http-nio-8003"_Processin...  7.856477e+09  7.856477e+09  7.856466e+09  7.856456e+09     31
879    Redis01        OSLinux-OSLinux_NETWORK_ens160_NETPacketsIn  4.858536e+09  4.858498e+09  1.170000e+02  7.550000e+01     15
880    Redis01       OSLinux-OSLinux_NETWORK_ens160_NETPacketsOut  4.385705e+09  4.385677e+09  8.500000e+01  4.850000e+01     15
1016   Redis02                redis-Redis_6379_Redis  (maxmemory)  4.294967e+09  4.294967e+09  4.294967e+09  4.294967e+09     30
...        ...                                                ...           ...           ...           ...           ...    ...
2         IG01          JVM-Memory_7778_JVM_Memory_HeapMemoryUsed  4.085646e+08  4.001423e+08  3.479865e+08  2.802512e+08     31
166       IG02        OSLinux-OSLinux_NETWORK_ens160_NETPacketsIn  3.598232e+08  3.598220e+08  6.000000e+00  4.000000e+00     15
76        IG01        OSLinux-OSLinux_NETWORK_ens160_NETPacketsIn  3.587183e+08  3.587170e+08  7.000000e+00  4.000000e+00     15
167       IG02       OSLinux-OSLinux_NETWORK_ens160_NETPacketsOut  3.345132e+08  3.345122e+08  5.000000e+00  3.000000e+00     15
77        IG01       OSLinux-OSLinux_NETWORK_ens160_NETPacketsOut  3.334989e+08  3.334979e+08  6.000000e+00  3.000000e+00     15

[50 rows x 7 columns],         file  rows    min_timestamp_shanghai    max_timestamp_shanghai
0  trace.csv  4155 2021-03-04 05:30:00+08:00 2021-03-04 05:59:00+08:00, 0         IG01
1         IG02
2         MG01
3         MG02
4     Tomcat01
5     Tomcat02
6     Tomcat03
7     Tomcat04
8     dockerA1
9     dockerA2
10    dockerB1
11    dockerB2
dtype: object,                        trace_name  count
40        trace.self.duration_p95    302
41           trace.self.row_count    302
39       trace.self.duration_mean    302
42    trace.to_MG01.duration_mean    117
44        trace.to_MG01.row_count    117
43     trace.to_MG01.duration_p95    117
46     trace.to_MG02.duration_p95    116
45    trace.to_MG02.duration_mean    116
47        trace.to_MG02.row_count    116
2       trace.from_IG01.row_count     87
1    trace.from_IG01.duration_p95     87
0   trace.from_IG01.duration_mean     87
3   trace.from_IG02.duration_mean     82
4    trace.from_IG02.duration_p95     82
5       trace.from_IG02.row_count     82
6   trace.from_MG01.duration_mean     64
7    trace.from_MG01.duration_p95     64
8       trace.from_MG01.row_count     64
10   trace.from_MG02.duration_p95     64
9   trace.from_MG02.duration_mean     64,       file  rows    min_timestamp_shanghai    max_timestamp_shanghai
0  log.csv   386 2021-03-04 05:30:00+08:00 2021-03-04 05:59:00+08:00, 0    Tomcat01
1    Tomcat02
2    Tomcat03
3    Tomcat04
4    apache01
5    apache02
dtype: object,            log_name  count
0   log.error_count    161
2     log.row_count    161
1  log.gc_pause_max     64,              file  rows min_timestamp_shanghai max_timestamp_shanghai
0  error_logs.csv     0                    NaT                    NaT, Series([], dtype: object))

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

# Reuse variables from previous execution: metric, metric_agg, to_shanghai_ts, tz
import pandas as pd
import numpy as np

# Ensure timestamp in Shanghai tz on metric (reuse if already present)
if 'ts_sh' not in metric.columns:
    metric['ts_sh'] = to_shanghai_ts(metric['timestamp'])

# Define window in Asia/Shanghai (UTC+8)
window_start = pd.Timestamp("2021-03-04 05:30:00").tz_localize(tz)
window_end   = pd.Timestamp("2021-03-04 06:00:00").tz_localize(tz)

# Filter metric to the failure window (inclusive)
metric_window = metric[(metric['ts_sh'] >= window_start) & (metric['ts_sh'] <= window_end)].copy()

# Prepare global thresholds (use metric_agg computed earlier which contains P95 and P5)
thresholds = metric_agg[['cmdb_id','kpi_name','P95','P5']].copy()

# Merge thresholds into windowed metric rows
merged = metric_window.merge(thresholds, on=['cmdb_id','kpi_name'], how='left', validate='m:1')

# Drop rows without global thresholds (should be rare); keep only pairs with thresholds
merged = merged.dropna(subset=['P95','P5'])

# Aggregation function per (cmdb_id, kpi_name)
def agg_fn(g):
    total_count = int(g.shape[0])
    max_value = float(g['value'].max())
    min_value = float(g['value'].min())
    P95 = float(g['P95'].iloc[0])
    P5  = float(g['P5'].iloc[0])
    high_mask = g['value'] > P95
    low_mask  = g['value'] < P5
    high_anom_count = int(high_mask.sum())
    low_anom_count  = int(low_mask.sum())
    first_high_ts = (g.loc[high_mask, 'ts_sh'].min() if high_anom_count>0 else pd.NaT)
    first_low_ts  = (g.loc[low_mask,  'ts_sh'].min() if low_anom_count>0  else pd.NaT)
    severity_high = (max_value / P95) if (P95 != 0) else np.nan
    severity_low  = (P5 / min_value) if (min_value != 0) else np.nan

    return pd.Series({
        'total_count': total_count,
        'high_anom_count': high_anom_count,
        'first_high_anom_ts': first_high_ts,
        'max_value_in_window': max_value,
        'P95': P95,
        'severity_high': severity_high,
        'low_anom_count': low_anom_count,
        'first_low_anom_ts': first_low_ts,
        'min_value_in_window': min_value,
        'P5': P5,
        'severity_low': severity_low
    })

agg = merged.groupby(['cmdb_id','kpi_name']).apply(agg_fn).reset_index()

# Keep only rows with at least one high or low anomaly
agg_filtered = agg[(agg['high_anom_count']>0) | (agg['low_anom_count']>0)].copy()

# Compute max severity for sorting, handle NaN
agg_filtered['severity_max'] = agg_filtered[['severity_high','severity_low']].replace([np.inf, -np.inf], np.nan).fillna(0).max(axis=1)

# Format timestamps as ISO strings (keep timezone info); NULL if NaT
agg_filtered['first_high_anom_ts'] = agg_filtered['first_high_anom_ts'].apply(lambda x: x.isoformat() if pd.notna(x) else None)
agg_filtered['first_low_anom_ts']  = agg_filtered['first_low_anom_ts'].apply(lambda x: x.isoformat() if pd.notna(x) else None)

# Round numeric columns for compactness
for c in ['max_value_in_window','min_value_in_window','P95','P5','severity_high','severity_low','severity_max']:
    agg_filtered[c] = agg_filtered[c].round(6)

# Select and order columns as requested
out_cols = ['cmdb_id','kpi_name','total_count',
            'high_anom_count','first_high_anom_ts','max_value_in_window','P95','severity_high',
            'low_anom_count','first_low_anom_ts','min_value_in_window','P5','severity_low','severity_max']

result = agg_filtered.sort_values('severity_max', ascending=False)[out_cols].head(20)

# Display compact result (top 20)
result

```
Out[2]:
```
summary_top_n = 5
top = result.sort_values('severity_max', ascending=False).head(summary_top_n)

lines = []
lines.append(f"Anomaly detection summary for window 2021-03-04 05:30:00+08:00 to 2021-03-04 06:00:00+08:00 (showing top {summary_top_n} by severity):")
lines.append(f"Total anomalous (high or low) metric series found in window: {result.shape[0]}")
lines.append("")

for i, r in top.reset_index(drop=True).iterrows():
    lines.append(f"{i+1}. {r['cmdb_id']} / {r['kpi_name']}")
    lines.append(f"   - points in window: {r['total_count']}")
    lines.append(f"   - high anomalies: {r['high_anom_count']}, first_high: {r['first_high_anom_ts']}, P95={r['P95']}, max_in_window={r['max_value_in_window']}, severity_high={r['severity_high']}")
    lines.append(f"   - low anomalies:  {r['low_anom_count']}, first_low:  {r['first_low_anom_ts']}, P5={r['P5']}, min_in_window={r['min_value_in_window']}, severity_low={r['severity_low']}")
    lines.append(f"   - overall severity score: {r['severity_max']}")
    lines.append("")

lines.append("Notable findings:")
lines.append("- Highest severity: Redis02 OSLinux-CPU_CPU_CPUWio with severity ≈ 52.03 (2 high-anomaly points).")
lines.append("- Several CPU I/O and MySQL/LocalDisk KPIs show high severity ratios (severity 20-50 range), indicating strong spikes above their global P95.")
lines.append("- Many series exhibit only high anomalies (no low anomalies) or low anomalies only; results list includes both types.")
lines.append("")
lines.append("If you want the full top-20 table or details for any specific (cmdb_id, kpi_name), ask to display it by name or index.")

summary = "\n".join(lines)
summary

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name  total_count  high_anom_count         first_high_anom_ts  max_value_in_window         P95  severity_high  low_anom_count          first_low_anom_ts  min_value_in_window         P5  severity_low  severity_max
933    Redis02                             OSLinux-CPU_CPU_CPUWio           31                2  2021-03-04T05:39:00+08:00             0.864400    0.016612      52.034674               1  2021-03-04T05:50:00+08:00             0.004155   0.004156      1.000241     52.034674
105       IG02                             OSLinux-CPU_CPU_CPUWio           31                2  2021-03-04T05:43:00+08:00             1.263245    0.024940      50.652379               1  2021-03-04T05:51:00+08:00             0.004155   0.004200      1.010830     50.652379
199       MG01                             OSLinux-CPU_CPU_CPUWio           21                1  2021-03-04T06:00:00+08:00             0.764700    0.033200      23.033133               0                       None             0.004200   0.004200      1.000000     23.033133
374    Mysql01                        Mysql-MySQL_3306_Com Insert           20                1  2021-03-04T05:31:00+08:00             0.266700    0.013335      20.000000               0                       None             0.000000   0.000000           NaN     20.000000
587    Mysql02                   Mysql-MySQL_3306_Aborted Clients           20                1  2021-03-04T05:36:00+08:00             0.016700    0.000835      20.000000               0                       None             0.000000   0.000000           NaN     20.000000
1143  Tomcat01       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT           20                1  2021-03-04T05:51:00+08:00             1.000000    0.050000      20.000000               0                       None             0.000000   0.000000           NaN     20.000000
291       MG02                             OSLinux-CPU_CPU_CPUWio           21                1  2021-03-04T06:00:00+08:00             0.627500    0.033200      18.900602               1  2021-03-04T05:37:00+08:00             0.004200   0.008300      1.976190     18.900602
1522  apache01                        OSLinux-CPU_CPU_CPUUserTime           30                2  2021-03-04T05:54:00+08:00             2.204000    0.191817      11.490121               2  2021-03-04T05:55:00+08:00             0.070572   0.083021      1.176397     11.490121
427    Mysql01         Mysql-MySQL_3306_Innodb log write requests           20                1  2021-03-04T05:31:00+08:00             1.966700    0.177470      11.081873               0                       None             0.000000   0.000000           NaN     11.081873
788    Mysql02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdc_DSKWTps           15                1  2021-03-04T05:47:00+08:00            18.750000   15.903310       1.179000               1  2021-03-04T05:51:00+08:00             0.100000   0.893310      8.933100      8.933100
789    Mysql02   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdc_DSKWrite           15                1  2021-03-04T05:47:00+08:00           491.333300  384.839990       1.276721               1  2021-03-04T05:51:00+08:00             2.133333  16.693310      7.824990      7.824990
15        IG01                             OSLinux-CPU_CPU_CPUWio           31                2  2021-03-04T05:41:00+08:00             0.203674    0.031200       6.528013               1  2021-03-04T05:32:00+08:00             0.004200   0.008300      1.976190      6.528013
901    Redis01  redis-Redis_6379_Redis  (instantaneous_ops_per...           31                2  2021-03-04T05:38:00+08:00           320.000000   66.000000       4.848485               0                       None             0.000000   0.000000           NaN      4.848485
367    Mysql01                  Mysql-MySQL_3306_Binlog Cache Use           20                1  2021-03-04T05:31:00+08:00             0.400000    0.083365       4.798177               0                       None             0.000000   0.000000           NaN      4.798177
395    Mysql01                  Mysql-MySQL_3306_Handler Read Rnd           20                1  2021-03-04T05:54:00+08:00             3.133300    0.679165       4.613459               0                       None             0.000000   0.000000           NaN      4.613459
561    Mysql01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdc_DSKRTps           16                1  2021-03-04T06:00:00+08:00             0.016667    0.004167       4.000000               0                       None             0.000000   0.000000           NaN      4.000000
562    Mysql01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdc_DSKRead           16                1  2021-03-04T06:00:00+08:00             1.066667    0.266667       4.000000               0                       None             0.000000   0.000000           NaN      4.000000
1562  apache01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKPer...           16                1  2021-03-04T05:32:00+08:00             2.000000    0.500000       4.000000               0                       None             0.000000   0.000000           NaN      4.000000
560    Mysql01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdc_DSKPer...           16                1  2021-03-04T05:30:00+08:00             0.400000    0.100000       4.000000               0                       None             0.000000   0.000000           NaN      4.000000
433    Mysql01             Mysql-MySQL_3306_Innodb os log written           20                1  2021-03-04T05:31:00+08:00          1126.400000  299.520000       3.760684               0                       None             0.000000   0.000000           NaN      3.760684```
```

In [ ]:
# In[3]:

# Reuse existing variables: metric, metric_agg, result, metric_window, to_shanghai_ts, tz
import pandas as pd
import numpy as np

# Ensure metric_window exists (window: 2021-03-04 05:30:00+08 to 2021-03-04 06:00:00+08)
window_start = pd.Timestamp("2021-03-04 05:30:00").tz_localize(tz)
window_end   = pd.Timestamp("2021-03-04 06:00:00").tz_localize(tz)
if 'metric_window' not in globals():
    metric['ts_sh'] = to_shanghai_ts(metric['timestamp'])
    metric_window = metric[(metric['ts_sh'] >= window_start) & (metric['ts_sh'] <= window_end)].copy()

# Top 10 anomalous series from previous 'result' by severity_max
top10_pairs = result.sort_values('severity_max', ascending=False).head(10)[['cmdb_id','kpi_name']].drop_duplicates().reset_index(drop=True)

# Prepare threshold lookup from metric_agg (global thresholds P95, P5)
thresh = metric_agg.set_index(['cmdb_id','kpi_name'])[['P95','P5']]

runs = []
# Iterate top10 pairs and detect consecutive runs for high and low anomalies
for _, row in top10_pairs.iterrows():
    cmdb = row['cmdb_id']
    kpi = row['kpi_name']
    # get global thresholds
    try:
        P95 = float(thresh.loc[(cmdb,kpi),'P95'])
        P5  = float(thresh.loc[(cmdb,kpi),'P5'])
    except KeyError:
        # skip if no global thresholds
        continue

    # extract series rows in the window
    s = metric_window[(metric_window['cmdb_id']==cmdb) & (metric_window['kpi_name']==kpi)].sort_values('timestamp').copy()
    if s.empty:
        continue

    # Ensure ts_sh exists
    if 'ts_sh' not in s.columns:
        s['ts_sh'] = to_shanghai_ts(s['timestamp'])

    for run_type in ['high','low']:
        if run_type == 'high':
            mask = s['value'] > P95
        else:
            mask = s['value'] < P5

        sf = s[mask].copy()
        if sf.empty:
            continue

        # detect consecutive runs where timestamp difference == 60
        sf = sf.sort_values('timestamp').reset_index(drop=True)
        # gap for first row should start a new run
        gaps = sf['timestamp'].diff().fillna(999)
        new_run = gaps != 60
        run_id = new_run.cumsum()
        sf['run_id'] = run_id

        # aggregate runs
        grp = sf.groupby('run_id')
        for rid, g in grp:
            start_ts = g['ts_sh'].iloc[0]
            end_ts = g['ts_sh'].iloc[-1]
            length_minutes = int(g.shape[0])
            max_value = float(g['value'].max())
            min_value = float(g['value'].min())
            severity_high = None
            severity_low = None
            if run_type == 'high':
                severity_high = (max_value / P95) if (P95 != 0) else np.nan
            else:
                severity_low = (P5 / min_value) if (min_value != 0) else np.nan

            significant = False
            if run_type == 'high':
                significant = (severity_high is not None) and (not np.isnan(severity_high)) and (severity_high >= 1.5)
            else:
                significant = (severity_low is not None) and (not np.isnan(severity_low)) and (severity_low >= 1.5)

            runs.append({
                'cmdb_id': cmdb,
                'kpi_name': kpi,
                'run_type': run_type,
                'start_ts': start_ts,
                'end_ts': end_ts,
                'length_minutes': length_minutes,
                'max_value': max_value,
                'min_value': min_value,
                'P95': P95,
                'P5': P5,
                'severity_high': (round(severity_high,6) if (severity_high is not None and not np.isnan(severity_high)) else None),
                'severity_low':  (round(severity_low,6)  if (severity_low is not None and not np.isnan(severity_low)) else None),
                'significant': significant
            })

# Build DataFrame of runs
runs_df = pd.DataFrame(runs)

# Totals before filtering
total_runs_before = len(runs_df)

if runs_df.empty:
    runs_out = runs_df  # empty
    remaining_count = 0
else:
    # Exclude runs that are length == 1 AND not significant
    filtered = runs_df[~((runs_df['length_minutes']==1) & (runs_df['significant']==False))].copy()
    remaining_count = len(filtered)

    # Compute severity_max for sorting
    filtered['severity_high_f'] = filtered['severity_high'].fillna(0)
    filtered['severity_low_f'] = filtered['severity_low'].fillna(0)
    filtered['severity_max'] = filtered[['severity_high_f','severity_low_f']].max(axis=1)

    # Format timestamps as ISO strings (keep timezone)
    filtered['start_ts'] = filtered['start_ts'].apply(lambda x: x.isoformat() if pd.notna(x) else None)
    filtered['end_ts'] = filtered['end_ts'].apply(lambda x: x.isoformat() if pd.notna(x) else None)

    # Round numeric columns for compact display
    for c in ['max_value','min_value','P95','P5','severity_max']:
        if c in filtered.columns:
            filtered[c] = filtered[c].round(6)

    # Select order of columns requested
    out_cols = ['cmdb_id','kpi_name','run_type','start_ts','end_ts','length_minutes',
                'max_value','min_value','P95','P5','severity_high','severity_low','significant','severity_max']

    runs_out = filtered.sort_values('severity_max', ascending=False)[out_cols].head(20)

# Display compact outputs: runs_out (top runs) and counts (total before, remaining after)
runs_out, (total_runs_before, remaining_count)

```
Out[3]:
```
Summary of detected anomalous runs (window 2021-03-04 05:30 → 06:00, Asia/Shanghai UTC+8)

- Total runs detected before filtering: 19
- Runs remaining after filtering (removed isolated non-significant single-minute runs): 11

Top remaining runs (sorted by severity, showing the most important 10; timestamps are UTC+8):
1. Redis02 / OSLinux-CPU_CPU_CPUWio — high
   - start/end: 2021-03-04T05:55:00+08:00
   - length: 1 minute
   - max=0.8644, P95=0.016612, severity_high ≈ 52.0347 (significant)

2. IG02 / OSLinux-CPU_CPU_CPUWio — high
   - start/end: 2021-03-04T06:00:00+08:00
   - length: 1 minute
   - max=1.263245, P95=0.02494, severity_high ≈ 50.6524 (significant)

3. MG01 / OSLinux-CPU_CPU_CPUWio — high
   - start/end: 2021-03-04T06:00:00+08:00
   - length: 1 minute
   - max=0.7647, P95=0.0332, severity_high ≈ 23.0331 (significant)

4. Mysql01 / Mysql-MySQL_3306_Com Insert — high
   - start/end: 2021-03-04T05:31:00+08:00
   - length: 1 minute
   - max=0.2667, P95=0.013335, severity_high = 20.0 (significant)

5. Mysql02 / Mysql-MySQL_3306_Aborted Clients — high
   - start/end: 2021-03-04T05:36:00+08:00
   - length: 1 minute
   - max=0.0167, P95=0.000835, severity_high = 20.0 (significant)

6. Tomcat01 / OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT — high
   - start/end: 2021-03-04T05:51:00+08:00
   - length: 1 minute
   - max=1.0, P95=0.05, severity_high = 20.0 (significant)

7. MG02 / OSLinux-CPU_CPU_CPUWio — high
   - start/end: 2021-03-04T06:00:00+08:00
   - length: 1 minute
   - max=0.6275, P95=0.0332, severity_high ≈ 18.9006 (significant)

8. apache01 / OSLinux-CPU_CPU_CPUUserTime — high
   - start/end: 2021-03-04T05:56:00+08:00
   - length: 1 minute
   - max=2.204, P95=0.191817, severity_high ≈ 11.4901 (significant)

9. Mysql01 / Mysql-MySQL_3306_Innodb log write requests — high
   - start/end: 2021-03-04T05:31:00+08:00
   - length: 1 minute
   - max=1.9667, P95=0.17747, severity_high ≈ 11.0819 (significant)

10. Mysql02 / OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdc_DSKWTps — low
    - start/end: 2021-03-04T05:51:00+08:00
    - length: 1 minute
    - min=0.1, P5=0.89331, severity_low ≈ 8.9331 (significant)

Notes / observations:
- Most remaining runs are single-minute spikes but nevertheless marked "significant" because their severity ratio >= 1.5.
- The highest severities are CPU I/O (CPUWio) spikes on Redis02, IG02, MG01 with severity > 20, suggesting extreme deviation from global P95.
- There are both high (value > global P95) and low (value < global P5) runs; the strongest low-run example is disk write throughput on Mysql02 (severity ≈ 8.93).
- If you want the full run table (all remaining runs, or raw rows for any run) I can display it.

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                                         kpi_name run_type                   start_ts                     end_ts  length_minutes  max_value  min_value        P95        P5  severity_high  severity_low  significant  severity_max
1    Redis02                           OSLinux-CPU_CPU_CPUWio     high  2021-03-04T05:55:00+08:00  2021-03-04T05:55:00+08:00               1   0.864400   0.864400   0.016612  0.004156      52.034674           NaN         True     52.034674
4       IG02                           OSLinux-CPU_CPU_CPUWio     high  2021-03-04T06:00:00+08:00  2021-03-04T06:00:00+08:00               1   1.263245   1.263245   0.024940  0.004200      50.652379           NaN         True     50.652379
6       MG01                           OSLinux-CPU_CPU_CPUWio     high  2021-03-04T06:00:00+08:00  2021-03-04T06:00:00+08:00               1   0.764700   0.764700   0.033200  0.004200      23.033133           NaN         True     23.033133
7    Mysql01                      Mysql-MySQL_3306_Com Insert     high  2021-03-04T05:31:00+08:00  2021-03-04T05:31:00+08:00               1   0.266700   0.266700   0.013335  0.000000      20.000000           NaN         True     20.000000
8    Mysql02                 Mysql-MySQL_3306_Aborted Clients     high  2021-03-04T05:36:00+08:00  2021-03-04T05:36:00+08:00               1   0.016700   0.016700   0.000835  0.000000      20.000000           NaN         True     20.000000
9   Tomcat01     OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT     high  2021-03-04T05:51:00+08:00  2021-03-04T05:51:00+08:00               1   1.000000   1.000000   0.050000  0.000000      20.000000           NaN         True     20.000000
10      MG02                           OSLinux-CPU_CPU_CPUWio     high  2021-03-04T06:00:00+08:00  2021-03-04T06:00:00+08:00               1   0.627500   0.627500   0.033200  0.008300      18.900602           NaN         True     18.900602
13  apache01                      OSLinux-CPU_CPU_CPUUserTime     high  2021-03-04T05:56:00+08:00  2021-03-04T05:56:00+08:00               1   2.204000   2.204000   0.191817  0.083021      11.490121           NaN         True     11.490121
16   Mysql01       Mysql-MySQL_3306_Innodb log write requests     high  2021-03-04T05:31:00+08:00  2021-03-04T05:31:00+08:00               1   1.966700   1.966700   0.177470  0.000000      11.081873           NaN         True     11.081873
18   Mysql02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdc_DSKWTps      low  2021-03-04T05:51:00+08:00  2021-03-04T05:51:00+08:00               1   0.100000   0.100000  15.903310  0.893310            NaN       8.93310         True      8.933100
11      MG02                           OSLinux-CPU_CPU_CPUWio      low  2021-03-04T05:37:00+08:00  2021-03-04T05:37:00+08:00               1   0.004200   0.004200   0.033200  0.008300            NaN       1.97619         True      1.976190, (19, 11))```
```

In [ ]:
# In[4]:

# Reuse variables: trace, to_shanghai_ts, tz
import pandas as pd

# Ensure trace has Shanghai timezone column
if 'ts_sh' not in trace.columns:
    trace['ts_sh'] = to_shanghai_ts(trace['timestamp'])

# Define anomaly candidate events (cmdb_id and start timestamps in UTC+8)
events = [
    ('Redis02', '2021-03-04T05:55:00'),
    ('IG02',   '2021-03-04T06:00:00'),
    ('MG01',   '2021-03-04T06:00:00'),
    ('MG02',   '2021-03-04T06:00:00'),
    ('Mysql01','2021-03-04T05:31:00'),
    ('Mysql02','2021-03-04T05:36:00'),
    ('Mysql02','2021-03-04T05:51:00'),
    ('Tomcat01','2021-03-04T05:51:00'),
    ('apache01','2021-03-04T05:56:00'),
]

# Build a DataFrame of trace rows within ±60 seconds for each event
rows = []
for cmdb, ts_str in events:
    center = pd.Timestamp(ts_str).tz_localize(tz)
    start = center - pd.Timedelta(seconds=60)
    end = center + pd.Timedelta(seconds=60)
    sel = trace[(trace['cmdb_id'] == cmdb) & (trace['ts_sh'] >= start) & (trace['ts_sh'] <= end)].copy()
    if not sel.empty:
        sel['event_center'] = center  # mark which event window produced these rows
        rows.append(sel)

if rows:
    matched = pd.concat(rows).drop_duplicates().sort_values(['cmdb_id','ts_sh','trace_name']).reset_index(drop=True)
else:
    matched = trace.iloc[0:0].copy()  # empty frame with same columns

# Prepare compact output: select columns and format timestamp as ISO+08
if not matched.empty:
    matched['timestamp_iso'] = matched['ts_sh'].apply(lambda x: x.isoformat() if pd.notna(x) else None)
    out_df = matched[['cmdb_id','timestamp_iso','trace_name','value']].head(50)
    counts = matched.groupby('cmdb_id').size().reset_index(name='count').sort_values('count', ascending=False)
else:
    out_df = matched[['cmdb_id','timestamp','trace_name','value']].head(50)  # empty
    counts = pd.DataFrame(columns=['cmdb_id','count'])

# Return compact results: matched rows (limited) and counts per cmdb_id
out_df, counts

```
Out[4]:
```
I extracted trace rows within ±60 seconds of each specified anomaly start time. Results (compact):

- Total matched trace rows returned: 36 (limited to 50 as requested).
- Counts per component:
  - MG02: 12 rows
  - IG02: 9 rows
  - Tomcat01: 9 rows
  - MG01: 6 rows

- No trace rows were found within ±60s for these events:
  - Redis02 @ 2021-03-04T05:55:00+08:00
  - Mysql01 @ 2021-03-04T05:31:00+08:00
  - Mysql02 @ 2021-03-04T05:36:00+08:00 and 2021-03-04T05:51:00+08:00
  - apache01 @ 2021-03-04T05:56:00+08:00

Notable observations from the matched traces:
- IG02 (around 2021-03-04T05:59:00+08:00) shows trace.self.duration_mean ≈ 0.08725, trace.self.duration_p95 ≈ 0.29, and trace.self.row_count = 8; there are also traces to Tomcat01 (duration_mean/p95).
- Tomcat01 (near 05:52) has trace.self.duration_p95 ≈ 0.10725 and traces to MG01 (duration_mean/p95, row_count).
- MG02 produced the most trace entries in these windows (12 rows), indicating more trace coverage for that host at the checked times.
- Common trace types present: trace.self.duration_mean / duration_p95 / row_count and trace.to_* (e.g., trace.to_MG01, trace.to_MG02) duration and row_count metrics.

If you'd like, I can:
- Display the full matched table (all 36 rows) for inspection, or
- Re-run with a wider time window (e.g., ±5 minutes) to try to capture traces for Redis02, Mysql*, or apache01.

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id              timestamp_iso                       trace_name     value
0       IG02  2021-03-04T05:59:00+08:00         trace.self.duration_mean  0.087250
1       IG02  2021-03-04T05:59:00+08:00          trace.self.duration_p95  0.290000
2       IG02  2021-03-04T05:59:00+08:00             trace.self.row_count  8.000000
3       IG02  2021-03-04T05:59:00+08:00  trace.to_Tomcat01.duration_mean  0.018333
4       IG02  2021-03-04T05:59:00+08:00   trace.to_Tomcat01.duration_p95  0.018900
..       ...                        ...                              ...       ...
31  Tomcat01  2021-03-04T05:52:00+08:00          trace.self.duration_p95  0.107250
32  Tomcat01  2021-03-04T05:52:00+08:00             trace.self.row_count  6.000000
33  Tomcat01  2021-03-04T05:52:00+08:00      trace.to_MG01.duration_mean  0.118000
34  Tomcat01  2021-03-04T05:52:00+08:00       trace.to_MG01.duration_p95  0.118000
35  Tomcat01  2021-03-04T05:52:00+08:00          trace.to_MG01.row_count  1.000000

[36 rows x 4 columns],     cmdb_id  count
2      MG02     12
0      IG02      9
3  Tomcat01      9
1      MG01      6)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[5]:

# Reuse existing variables: log, error_logs, to_shanghai_ts, tz
import pandas as pd

# Ensure Shanghai timezone timestamps exist
if 'ts_sh' not in log.columns:
    log['ts_sh'] = to_shanghai_ts(log['timestamp'])
if 'ts_sh' not in error_logs.columns:
    # error_logs may be empty; handle gracefully
    if 'timestamp' in error_logs.columns and not error_logs.empty:
        error_logs['ts_sh'] = to_shanghai_ts(error_logs['timestamp'])
    else:
        error_logs['ts_sh'] = pd.Series(dtype='datetime64[ns, Asia/Shanghai]')

# Define components and window (inclusive) in Asia/Shanghai (UTC+8)
components = ['Tomcat01','apache01']
window_start = pd.Timestamp("2021-03-04 05:45:00").tz_localize(tz)
window_end   = pd.Timestamp("2021-03-04 06:00:00").tz_localize(tz)

# Filter log rows for components and window
matched = log[(log['cmdb_id'].isin(components)) & (log['ts_sh'] >= window_start) & (log['ts_sh'] <= window_end)].copy()

# 1) total number of matched rows
total_matched = int(matched.shape[0])

# 2) counts per cmdb_id
counts_by_cmdb = matched.groupby('cmdb_id').size().reset_index(name='count').sort_values('count', ascending=False)

# 3) counts per log_name (top entries)
counts_by_logname = matched.groupby('log_name').size().reset_index(name='count').sort_values('count', ascending=False).head(20)

# 4) matched rows limited to first 50, with requested columns and timestamp ISO+08, sorted by timestamp ascending
if not matched.empty:
    matched = matched.sort_values('ts_sh', ascending=True).reset_index(drop=True)
    matched['timestamp_iso'] = matched['ts_sh'].apply(lambda x: x.isoformat() if pd.notna(x) else None)
    matched_rows_out = matched[['timestamp_iso','cmdb_id','log_name','value']].rename(columns={'timestamp_iso':'timestamp'}).head(50)
else:
    matched_rows_out = matched[['timestamp','cmdb_id','log_name','value']].head(0)

# Check error_logs for those components/times: return count only
if not error_logs.empty:
    err_matched = error_logs[(error_logs['cmdb_id'].isin(components)) & (error_logs['ts_sh'] >= window_start) & (error_logs['ts_sh'] <= window_end)]
    error_log_count = int(err_matched.shape[0])
else:
    error_log_count = 0

# Compact outputs
total_matched, counts_by_cmdb, counts_by_logname, matched_rows_out, error_log_count

```
Out[5]:
```
summary = (
    "Log query summary for Tomcat01 and apache01 between 2021-03-04 05:45:00+08:00 and 2021-03-04 06:00:00+08:00:\n\n"
    "1) Total matched rows: 71\n\n"
    "2) Counts per cmdb_id:\n"
    "   - Tomcat01: 45\n"
    "   - apache01: 26\n\n"
    "3) Top log_name counts (within matched rows):\n"
    "   - log.error_count: 28\n"
    "   - log.row_count: 28\n"
    "   - log.gc_pause_max: 15\n\n"
    "4) Matched rows: first 50 rows are available in the kernel variable 'matched_rows_out' (columns: timestamp, cmdb_id, log_name, value), sorted by timestamp ascending.\n\n"
    "Error logs check: error_logs.csv contains 0 entries for these components in the same time window.\n\n"
    "If you want to view the first 50 matched rows here, ask and I will display 'matched_rows_out'."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(71,     cmdb_id  count
0  Tomcat01     45
1  apache01     26,            log_name  count
0   log.error_count     28
2     log.row_count     28
1  log.gc_pause_max     15,                     timestamp   cmdb_id          log_name   value
0   2021-03-04T05:45:00+08:00  Tomcat01   log.error_count    0.00
1   2021-03-04T05:45:00+08:00  Tomcat01  log.gc_pause_max    5.13
2   2021-03-04T05:45:00+08:00  Tomcat01     log.row_count   74.00
3   2021-03-04T05:45:00+08:00  apache01   log.error_count    0.00
4   2021-03-04T05:45:00+08:00  apache01     log.row_count    9.00
..                        ...       ...               ...     ...
45  2021-03-04T05:54:00+08:00  Tomcat01     log.row_count  195.00
46  2021-03-04T05:55:00+08:00  Tomcat01   log.error_count    0.00
47  2021-03-04T05:55:00+08:00  Tomcat01  log.gc_pause_max    2.09
48  2021-03-04T05:55:00+08:00  Tomcat01     log.row_count  200.00
49  2021-03-04T05:55:00+08:00  apache01   log.error_count    0.00

[50 rows x 4 columns], 0)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[6]:

# Reuse existing variables: log, to_shanghai_ts, tz
import pandas as pd

# Ensure Shanghai timezone timestamps exist on log
if 'ts_sh' not in log.columns:
    log['ts_sh'] = to_shanghai_ts(log['timestamp'])

# Define Tomcat01 small window (inclusive) in Asia/Shanghai (UTC+8)
start = pd.Timestamp("2021-03-04 05:50:00").tz_localize(tz)
end   = pd.Timestamp("2021-03-04 05:56:00").tz_localize(tz)

# Filter log rows for Tomcat01 in the window
tomcat_window = log[(log['cmdb_id']=='Tomcat01') & (log['ts_sh'] >= start) & (log['ts_sh'] <= end)].copy()

# 1) total matched rows count
total_count = int(tomcat_window.shape[0])

# 2) counts per log_name within that window
counts_by_logname = tomcat_window.groupby('log_name').size().reset_index(name='count').sort_values('count', ascending=False)

# 3) matched rows limited to first 50 sorted by timestamp ascending, with ISO+08 timestamps
if not tomcat_window.empty:
    tomcat_window = tomcat_window.sort_values('ts_sh', ascending=True).reset_index(drop=True)
    tomcat_window['timestamp_iso'] = tomcat_window['ts_sh'].apply(lambda x: x.isoformat() if pd.notna(x) else None)
    matched_rows_out = tomcat_window[['timestamp_iso','cmdb_id','log_name','value']].rename(columns={'timestamp_iso':'timestamp'}).head(50)
else:
    matched_rows_out = tomcat_window[['timestamp','cmdb_id','log_name','value']].head(0)

# Return compact results
total_count, counts_by_logname, matched_rows_out

```
Out[6]:
```
# Summarize Tomcat01 log rows between 2021-03-04 05:50 and 05:56 (UTC+8)
# Reuse kernel variables: total_count, counts_by_logname, matched_rows_out

import pandas as pd

if 'matched_rows_out' in globals() and not matched_rows_out.empty:
    earliest = matched_rows_out['timestamp'].min()
    latest = matched_rows_out['timestamp'].max()
    # counts_by_logname may exist; otherwise compute
    if 'counts_by_logname' in globals() and not counts_by_logname.empty:
        counts_list = ", ".join([f"{r['log_name']}={int(r['count'])}" for _, r in counts_by_logname.iterrows()])
    else:
        cb = matched_rows_out.groupby('log_name').size().reset_index(name='count').sort_values('count', ascending=False)
        counts_list = ", ".join([f"{r['log_name']}={int(r['count'])}" for _, r in cb.iterrows()])
    # quick observations
    def get_stats(name):
        df = matched_rows_out[matched_rows_out['log_name']==name]
        if df.empty:
            return None
        vals = pd.to_numeric(df['value'], errors='coerce')
        return vals.min(), vals.max()
    gc_stats = get_stats('log.gc_pause_max')
    rowcount_max = matched_rows_out[matched_rows_out['log_name']=='log.row_count']['value'].max() if 'log.row_count' in matched_rows_out['log_name'].values else None
    err_vals = matched_rows_out[matched_rows_out['log_name']=='log.error_count']['value'].unique().tolist()
    
    summary = (
        f"Tomcat01 logs 2021-03-04 05:50 → 05:56 (UTC+8):\n"
        f"1) Total matched rows: {int(total_count)}\n"
        f"2) Counts per log_name: {counts_list}\n"
        f"3) Time range: earliest={earliest}, latest={latest}\n"
        f"4) Quick observations:\n"
        f"   - log.gc_pause_max range in window: {gc_stats[0]} → {gc_stats[1]} (min → max) \n"
        f"   - max log.row_count observed: {rowcount_max}\n"
        f"   - log.error_count distinct values in window: {err_vals}\n\n"
        f"Matched rows (first 50) are available in kernel variable 'matched_rows_out' (columns: timestamp, cmdb_id, log_name, value)."
    )
else:
    summary = "No matched Tomcat01 rows found in the specified window."

summary

The original code execution output of IPython Kernel is also provided below for reference:

(21,            log_name  count
0   log.error_count      7
1  log.gc_pause_max      7
2     log.row_count      7,                     timestamp   cmdb_id          log_name   value
0   2021-03-04T05:50:00+08:00  Tomcat01   log.error_count    0.00
1   2021-03-04T05:50:00+08:00  Tomcat01  log.gc_pause_max    5.11
2   2021-03-04T05:50:00+08:00  Tomcat01     log.row_count   70.00
3   2021-03-04T05:51:00+08:00  Tomcat01   log.error_count    0.00
4   2021-03-04T05:51:00+08:00  Tomcat01  log.gc_pause_max    5.11
..                        ...       ...               ...     ...
16  2021-03-04T05:55:00+08:00  Tomcat01  log.gc_pause_max    2.09
17  2021-03-04T05:55:00+08:00  Tomcat01     log.row_count  200.00
18  2021-03-04T05:56:00+08:00  Tomcat01   log.error_count    0.00
19  2021-03-04T05:56:00+08:00  Tomcat01  log.gc_pause_max    2.10
20  2021-03-04T05:56:00+08:00  Tomcat01     log.row_count  192.00

[21 rows x 4 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[7]:

# Reuse existing variables where possible: trace, to_shanghai_ts, tz
import pandas as pd

# Ensure trace has Shanghai timezone column
if 'ts_sh' not in trace.columns:
    trace['ts_sh'] = to_shanghai_ts(trace['timestamp'])

# Define components and window (inclusive) in Asia/Shanghai (UTC+8)
components = ['Tomcat01', 'MG01']
start = pd.Timestamp("2021-03-04 05:50:00").tz_localize(tz)
end   = pd.Timestamp("2021-03-04 05:56:00").tz_localize(tz)

# Filter trace rows for the components and time window
matched_traces = trace[(trace['cmdb_id'].isin(components)) & (trace['ts_sh'] >= start) & (trace['ts_sh'] <= end)].copy()

# 1) total matched rows
total_matched = int(matched_traces.shape[0])

# 2) counts per cmdb_id
counts_by_cmdb = matched_traces.groupby('cmdb_id').size().reset_index(name='count').sort_values('count', ascending=False)

# 3) counts per trace_name (top entries)
counts_by_trace = matched_traces.groupby('trace_name').size().reset_index(name='count').sort_values('count', ascending=False).head(20)

# 4) per-timestamp summary limited to first 50 rows sorted by timestamp ascending
if not matched_traces.empty:
    matched_traces = matched_traces.sort_values(['ts_sh','trace_name']).reset_index(drop=True)
    matched_traces['timestamp_iso'] = matched_traces['ts_sh'].apply(lambda x: x.isoformat() if pd.notna(x) else None)
    per_timestamp = matched_traces[['timestamp_iso','cmdb_id','trace_name','value']].rename(columns={'timestamp_iso':'timestamp'}).head(50)
else:
    per_timestamp = matched_traces[['timestamp','cmdb_id','trace_name','value']].head(0)

# Additional: trace.self.row_count changes for Tomcat01 in the window
tomcat_rowcount = matched_traces[(matched_traces['cmdb_id']=='Tomcat01') & (matched_traces['trace_name']=='trace.self.row_count')].copy()
if not tomcat_rowcount.empty:
    tomcat_rowcount['timestamp_iso'] = tomcat_rowcount['ts_sh'].apply(lambda x: x.isoformat() if pd.notna(x) else None)
    tomcat_rowcount_changes = tomcat_rowcount[['timestamp_iso','value']].rename(columns={'timestamp_iso':'timestamp','value':'trace.self.row_count'}).reset_index(drop=True)
else:
    tomcat_rowcount_changes = tomcat_rowcount[['timestamp','value']].head(0)

# Return compact outputs (as per rules)
total_matched, counts_by_cmdb, counts_by_trace, per_timestamp, tomcat_rowcount_changes

```
Out[7]:
```
summary = (
    "Trace query summary for Tomcat01 and MG01 between 2021-03-04 05:50:00 and 2021-03-04 05:56:00 (UTC+8):\n\n"
    "1) Total matched rows: 189\n\n"
    "2) Counts per component:\n"
    "   - MG01: 135 rows\n"
    "   - Tomcat01: 54 rows\n\n"
    "3) Top trace_name counts (examples):\n"
    "   - trace.self.duration_p95: 9\n"
    "   - trace.self.row_count: 9\n"
    "   - trace.self.duration_mean: 9\n"
    "   - several trace.from_/trace.to_* metrics also appear (counts up to 5)\n\n"
    "4) Per-timestamp trace rows (sample, first 50 rows available):\n"
    "   - Results include timestamps like 2021-03-04T05:50:00+08:00, 05:51, 05:52, 05:53, 05:55, 05:56 with various trace.* metrics\n"
    "   - Full first-50-row table is available in the kernel variable 'per_timestamp'.\n\n"
    "Additional observation — Tomcat01 trace.self.row_count changes in the window:\n"
    "   - 2021-03-04T05:52:00+08:00  => 6.0\n"
    "   - 2021-03-04T05:53:00+08:00  => 12.0\n"
    "   - 2021-03-04T05:55:00+08:00  => 39.0\n"
    "   - 2021-03-04T05:56:00+08:00  => 24.0\n\n"
    "If you want to inspect the exact matched rows or the full per-timestamp table, request the kernel variable 'per_timestamp' or ask to show more rows."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(189,     cmdb_id  count
0      MG01    135
1  Tomcat01     54,                            trace_name  count
25            trace.self.duration_p95      9
26               trace.self.row_count      9
24           trace.self.duration_mean      9
13   trace.from_Tomcat03.duration_p95      5
12  trace.from_Tomcat03.duration_mean      5
44        trace.to_dockerB2.row_count      5
42    trace.to_dockerB2.duration_mean      5
43     trace.to_dockerB2.duration_p95      5
14      trace.from_Tomcat03.row_count      5
3       trace.from_IG02.duration_mean      4
7    trace.from_Tomcat01.duration_p95      4
5           trace.from_IG02.row_count      4
4        trace.from_IG02.duration_p95      4
38        trace.to_dockerA2.row_count      4
41        trace.to_dockerB1.row_count      4
39    trace.to_dockerB1.duration_mean      4
11      trace.from_Tomcat02.row_count      4
6   trace.from_Tomcat01.duration_mean      4
10   trace.from_Tomcat02.duration_p95      4
9   trace.from_Tomcat02.duration_mean      4,                     timestamp   cmdb_id                       trace_name    value
0   2021-03-04T05:50:00+08:00      MG01  trace.to_dockerA1.duration_mean  0.06500
1   2021-03-04T05:50:00+08:00      MG01   trace.to_dockerA1.duration_p95  0.06500
2   2021-03-04T05:50:00+08:00      MG01      trace.to_dockerA1.row_count  1.00000
3   2021-03-04T05:50:00+08:00      MG01  trace.to_dockerA2.duration_mean  0.06300
4   2021-03-04T05:50:00+08:00      MG01   trace.to_dockerA2.duration_p95  0.08550
..                        ...       ...                              ...      ...
45  2021-03-04T05:52:00+08:00  Tomcat01      trace.to_MG01.duration_mean  0.11800
46  2021-03-04T05:52:00+08:00  Tomcat01       trace.to_MG01.duration_p95  0.11800
47  2021-03-04T05:52:00+08:00  Tomcat01          trace.to_MG01.row_count  1.00000
48  2021-03-04T05:52:00+08:00      MG01  trace.to_dockerA1.duration_mean  0.34250
49  2021-03-04T05:52:00+08:00      MG01   trace.to_dockerA1.duration_p95  0.58235

[50 rows x 4 columns],                    timestamp  trace.self.row_count
0  2021-03-04T05:52:00+08:00                   6.0
1  2021-03-04T05:53:00+08:00                  12.0
2  2021-03-04T05:55:00+08:00                  39.0
3  2021-03-04T05:56:00+08:00                  24.0)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[8]:

# Reuse existing kernel variables where possible: metric, to_shanghai_ts, tz
import pandas as pd
import pytz

# Ensure timezone variable
try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# Ensure metric has Shanghai timezone column for timestamps
if 'ts_sh' not in metric.columns:
    metric['ts_sh'] = to_shanghai_ts(metric['timestamp'])

# Define Redis02 window (inclusive) in Asia/Shanghai (UTC+8)
start = pd.Timestamp("2021-03-04 05:50:00").tz_localize(tz)
end   = pd.Timestamp("2021-03-04 06:00:00").tz_localize(tz)

# Filter metric rows for Redis02 in the time window
redis_rows = metric[(metric['cmdb_id']=='Redis02') & (metric['ts_sh'] >= start) & (metric['ts_sh'] <= end)].copy()

# 1) total matched rows
total_matched = int(redis_rows.shape[0])

# 2) unique kpi_name values with occurrence counts (top 20)
kpi_counts = redis_rows.groupby('kpi_name').size().reset_index(name='count').sort_values('count', ascending=False).head(20)

# 3) rows where kpi_name contains 'CPU_CPU_CPUWio' (or exact match) with requested columns, sorted and limited
mask_cpuwio = redis_rows['kpi_name'].str.contains('CPU_CPU_CPUWio', na=False)
cpu_rows = redis_rows[mask_cpuwio].copy()
if not cpu_rows.empty:
    cpu_rows['timestamp'] = cpu_rows['ts_sh'].apply(lambda x: x.isoformat() if pd.notna(x) else None)
    cpu_rows = cpu_rows.sort_values('ts_sh', ascending=True)
    cpu_rows = cpu_rows[['timestamp','cmdb_id','kpi_name','value']].head(50)
else:
    # empty DataFrame with requested columns
    cpu_rows = pd.DataFrame(columns=['timestamp','cmdb_id','kpi_name','value'])

# Compact outputs
total_matched, kpi_counts, cpu_rows

```
Out[8]:
```
Summary for Redis02 metrics between 2021-03-04 05:50:00+08:00 and 2021-03-04 06:00:00+08:00:

1) Total matched rows: 807

2) Top unique kpi_name values (occurrence counts, up to 20 shown):
- OSLinux-CPU_CPU_CPUWio — 11
- OSLinux-CPU_CPU_CPULoad — 11
- OSLinux-CPU_CPU_CPUidleutil — 11
- OSLinux-CPU_CPU_CPUSysTime — 11
- OSLinux-CPU_CPU_CPUUserTime — 11
- redis-Redis_6379_redis server — 11
- redis-Redis_6379_Redis  (used_cpu_user_children) — 11
- redis-Redis_6379_Redis  (used_cpu_user) — 11
- redis-Redis_6379_Redis  (used_cpu_sys_children) — 11
- redis-Redis_6379_Redis  (used_memory_rss) — 11
- redis-Redis_6379_Redis  (aof_rewrite_scheduled) — 11
- redis-Redis_6379_Redis  (blocked_clients) — 11
- redis-Redis_6379_Redis  (client_biggest_input_buf) — 11
- redis-Redis_6379_Redis  (client_longest_output_list) — 11
- redis-Redis_6379_Redis  (connected_clients) — 11
- redis-Redis_6379_Redis  (connected_slaves) — 11
- redis-Redis_6379_Redis  (evicted_keys) — 11
- redis-Redis_6379_Redis  (expired_keys) — 11
- redis-Redis_6379_Redis  (instantaneous_ops_per_sec) — 11
- redis-Redis_6379_Redis  (keyspace_hits) — 11

(Many Redis-specific KPIs each appear 11 times in the window.)

3) Rows for kpi_name containing "CPU_CPU_CPUWio" (all matches, sorted by timestamp; times are ISO+08):
- 2021-03-04T05:50:00+08:00 — Redis02 — OSLinux-CPU_CPU_CPUWio — 0.004155
- 2021-03-04T05:51:00+08:00 — Redis02 — OSLinux-CPU_CPU_CPUWio — 0.004158
- 2021-03-04T05:52:00+08:00 — Redis02 — OSLinux-CPU_CPU_CPUWio — 0.004200
- 2021-03-04T05:53:00+08:00 — Redis02 — OSLinux-CPU_CPU_CPUWio — 0.008313
- 2021-03-04T05:54:00+08:00 — Redis02 — OSLinux-CPU_CPU_CPUWio — 0.004156
- 2021-03-04T05:55:00+08:00 — Redis02 — OSLinux-CPU_CPU_CPUWio — 0.864400  ← notable spike
- 2021-03-04T05:56:00+08:00 — Redis02 — OSLinux-CPU_CPU_CPUWio — 0.008315
- 2021-03-04T05:57:00+08:00 — Redis02 — OSLinux-CPU_CPU_CPUWio — 0.008312
- 2021-03-04T05:58:00+08:00 — Redis02 — OSLinux-CPU_CPU_CPUWio — 0.004200
- 2021-03-04T05:59:00+08:00 — Redis02 — OSLinux-CPU_CPU_CPUWio — 0.004156
- 2021-03-04T06:00:00+08:00 — Redis02 — OSLinux-CPU_CPU_CPUWio — 0.016600

Note: The CPU_WIO series is normally near ~0.004–0.008 in this window but shows a large spike to 0.864400 at 05:55:00+08:00.

The original code execution output of IPython Kernel is also provided below for reference:

(807,                                               kpi_name  count
8                               OSLinux-CPU_CPU_CPUWio     11
5                              OSLinux-CPU_CPU_CPULoad     11
9                          OSLinux-CPU_CPU_CPUidleutil     11
6                           OSLinux-CPU_CPU_CPUSysTime     11
7                          OSLinux-CPU_CPU_CPUUserTime     11
105                      redis-Redis_6379_redis server     11
101   redis-Redis_6379_Redis  (used_cpu_user_children)     11
100            redis-Redis_6379_Redis  (used_cpu_user)     11
99     redis-Redis_6379_Redis  (used_cpu_sys_children)     11
104          redis-Redis_6379_Redis  (used_memory_rss)     11
74     redis-Redis_6379_Redis  (aof_rewrite_scheduled)     11
75           redis-Redis_6379_Redis  (blocked_clients)     11
76   redis-Redis_6379_Redis  (client_biggest_input_...     11
77   redis-Redis_6379_Redis  (client_longest_output...     11
78         redis-Redis_6379_Redis  (connected_clients)     11
79          redis-Redis_6379_Redis  (connected_slaves)     11
80              redis-Redis_6379_Redis  (evicted_keys)     11
81              redis-Redis_6379_Redis  (expired_keys)     11
82   redis-Redis_6379_Redis  (instantaneous_ops_per...     11
83             redis-Redis_6379_Redis  (keyspace_hits)     11,                        timestamp  cmdb_id                kpi_name     value
18121  2021-03-04T05:50:00+08:00  Redis02  OSLinux-CPU_CPU_CPUWio  0.004155
19129  2021-03-04T05:51:00+08:00  Redis02  OSLinux-CPU_CPU_CPUWio  0.004158
20056  2021-03-04T05:52:00+08:00  Redis02  OSLinux-CPU_CPU_CPUWio  0.004200
20826  2021-03-04T05:53:00+08:00  Redis02  OSLinux-CPU_CPU_CPUWio  0.008313
21831  2021-03-04T05:54:00+08:00  Redis02  OSLinux-CPU_CPU_CPUWio  0.004156
22450  2021-03-04T05:55:00+08:00  Redis02  OSLinux-CPU_CPU_CPUWio  0.864400
23379  2021-03-04T05:56:00+08:00  Redis02  OSLinux-CPU_CPU_CPUWio  0.008315
24463  2021-03-04T05:57:00+08:00  Redis02  OSLinux-CPU_CPU_CPUWio  0.008312
25139  2021-03-04T05:58:00+08:00  Redis02  OSLinux-CPU_CPU_CPUWio  0.004200
26143  2021-03-04T05:59:00+08:00  Redis02  OSLinux-CPU_CPU_CPUWio  0.004156
27091  2021-03-04T06:00:00+08:00  Redis02  OSLinux-CPU_CPU_CPUWio  0.016600)```
```

In [ ]:
# In[9]:

# Reuse existing variables: metric, to_shanghai_ts, tz
import pandas as pd
import pytz

# Ensure tz is defined
try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# Ensure metric has Shanghai timezone column
if 'ts_sh' not in metric.columns:
    metric['ts_sh'] = to_shanghai_ts(metric['timestamp'])

# Target component and exact timestamps (UTC+8)
cmdb_target = 'Redis02'
ts_strings = ["2021-03-04 05:54:00", "2021-03-04 05:55:00", "2021-03-04 05:56:00"]
ts_list = [pd.Timestamp(s).tz_localize(tz) for s in ts_strings]

# Filter rows for Redis02 at those exact timestamps
matched = metric[(metric['cmdb_id'] == cmdb_target) & (metric['ts_sh'].isin(ts_list))].copy()

# Format output table: timestamp ISO+08, cmdb_id, kpi_name, value; sort by timestamp then kpi_name
if not matched.empty:
    matched['timestamp'] = matched['ts_sh'].apply(lambda x: x.isoformat() if pd.notna(x) else None)
    matched_out = matched.sort_values(['ts_sh','kpi_name'], ascending=[True, True])[
        ['timestamp','cmdb_id','kpi_name','value']
    ].head(200)  # limit to at most 200 rows
else:
    matched_out = matched[['timestamp','cmdb_id','kpi_name','value']].head(0)

# Counts per kpi_name for these timestamps (top 50)
kpi_counts = matched.groupby('kpi_name').size().reset_index(name='count').sort_values('count', ascending=False).head(50)

# Total matched rows
total_matched = int(matched.shape[0])

# Return compact results
matched_out, kpi_counts, total_matched

```
Out[9]:
```
Summary of Redis02 metric rows at timestamps 2021-03-04 05:54:00, 05:55:00, 05:56:00 (UTC+8):

1) Total matched rows: 246

2) Top kpi_name occurrence counts (sample, up to 20 shown):
- OSLinux-CPU_CPU_CPUSysTime — 3
- OSLinux-CPU_CPU_CPUUserTime — 3
- OSLinux-CPU_CPU_CPULoad — 3
- OSLinux-CPU_CPU_CPUWio — 3
- OSLinux-CPU_CPU_CPUidleutil — 3
- OSLinux-CPU_CPU-0_SingleCpuidle — 2
- OSLinux-CPU_CPU_CPUCpuUtil — 2
- OSLinux-CPU_CPU-2_SingleCpuidle — 2
- OSLinux-OSLinux_FILESYSTEM_-tmp_FSCapacity — 2
- OSLinux-OSLinux_FILESYSTEM_-tmp_FSAvailableSpace — 2
(There are many Redis-specific and OS KPIs; the full top-50 list is available in the kernel.)

3) Rows for these exact timestamps (sorted by timestamp then kpi_name):
- A compact table (up to 200 rows) was produced in the kernel with columns: timestamp (ISO+08), cmdb_id, kpi_name, value. Example rows include:
  - 2021-03-04T05:54:00+08:00 — Redis02 — OSLinux-CPU_CPU-0_SingleCpuidle — 99.534420
  - 2021-03-04T05:54:00+08:00 — Redis02 — OSLinux-CPU_CPU-2_SingleCpuidle — 99.185641
  - 2021-03-04T05:54:00+08:00 — Redis02 — OSLinux-CPU_CPU_CPUCpuUtil — 0.448900
  - ...
  - 2021-03-04T05:55:00+08:00 — Redis02 — OSLinux-CPU_CPU_CPUWio — 0.864400  (notable spike)
  - ...
  - 2021-03-04T05:56:00+08:00 — Redis02 — OSLinux-OSLinux_MEMORY_MEMORY_MEMFreeMem — 5488.000000

Notes:
- The CPU_WIO metric (OSLinux-CPU_CPU_CPUWio) appears at all three timestamps (3 rows) and includes the large spike at 05:55:00 (value 0.864400).
- The detailed table (up to 200 rows) and the full kpi_name counts (top 50) are stored in the kernel variables from the last query; tell me if you want that table printed in full or want further filtering/search.

The original code execution output of IPython Kernel is also provided below for reference:

(                       timestamp  cmdb_id                                          kpi_name        value
21825  2021-03-04T05:54:00+08:00  Redis02                   OSLinux-CPU_CPU-0_SingleCpuidle    99.534420
21826  2021-03-04T05:54:00+08:00  Redis02                   OSLinux-CPU_CPU-2_SingleCpuidle    99.185641
21827  2021-03-04T05:54:00+08:00  Redis02                        OSLinux-CPU_CPU_CPUCpuUtil     0.448900
21828  2021-03-04T05:54:00+08:00  Redis02                           OSLinux-CPU_CPU_CPULoad     0.000000
21829  2021-03-04T05:54:00+08:00  Redis02                        OSLinux-CPU_CPU_CPUSysTime     0.245220
...                          ...      ...                                               ...          ...
23421  2021-03-04T05:56:00+08:00  Redis02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKWrite     0.000000
23422  2021-03-04T05:56:00+08:00  Redis02            OSLinux-OSLinux_MEMORY_MEMORY_CacheMem   919.000000
23423  2021-03-04T05:56:00+08:00  Redis02          OSLinux-OSLinux_MEMORY_MEMORY_MEMFreeMem  5488.000000
23424  2021-03-04T05:56:00+08:00  Redis02      OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc    23.000000
23425  2021-03-04T05:56:00+08:00  Redis02      OSLinux-OSLinux_MEMORY_MEMORY_NoCacheMemPerc    10.716300

[200 rows x 4 columns],                                             kpi_name  count
6                         OSLinux-CPU_CPU_CPUSysTime      3
7                        OSLinux-CPU_CPU_CPUUserTime      3
5                            OSLinux-CPU_CPU_CPULoad      3
8                             OSLinux-CPU_CPU_CPUWio      3
9                        OSLinux-CPU_CPU_CPUidleutil      3
..                                               ...    ...
1                    OSLinux-CPU_CPU-0_SingleCpuidle      2
4                         OSLinux-CPU_CPU_CPUCpuUtil      2
3                    OSLinux-CPU_CPU-2_SingleCpuidle      2
31        OSLinux-OSLinux_FILESYSTEM_-tmp_FSCapacity      2
30  OSLinux-OSLinux_FILESYSTEM_-tmp_FSAvailableSpace      2

[50 rows x 2 columns], 246)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[10]:

# Reuse kernel variables where possible: metric, to_shanghai_ts, tz
import pandas as pd
import pytz

# Ensure timezone
try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# Ensure metric has Shanghai timezone column
if 'ts_sh' not in metric.columns:
    metric['ts_sh'] = to_shanghai_ts(metric['timestamp'])

# Target component and exact timestamp (UTC+8)
cmdb_target = 'Redis02'
target_ts = pd.Timestamp("2021-03-04 05:55:00").tz_localize(tz)

# Filter rows for Redis02 at the exact timestamp
matched = metric[(metric['cmdb_id'] == cmdb_target) & (metric['ts_sh'] == target_ts)].copy()

# 1) total matched rows
total_matched = int(matched.shape[0])

# 2) top kpi_name counts (up to 50)
kpi_counts = (
    matched.groupby('kpi_name')
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
    .head(50)
)

# 3) matched rows table: timestamp ISO+08, cmdb_id, kpi_name, value; sorted by kpi_name asc; limit 200
if not matched.empty:
    matched['timestamp'] = matched['ts_sh'].apply(lambda x: x.isoformat() if pd.notna(x) else None)
    matched_out = (
        matched.sort_values(['kpi_name', 'timestamp'], ascending=[True, True])
        [['timestamp','cmdb_id','kpi_name','value']]
        .head(200)
        .reset_index(drop=True)
    )
else:
    matched_out = matched[['timestamp','cmdb_id','kpi_name','value']].head(0)

# Additional: extract single values for requested Redis operational KPIs (NULL if not present)
patterns = {
    'blocked_clients': 'blocked_clients',
    'instantaneous_ops_per_sec': 'instantaneous_ops_per',
    'connected_clients': 'connected_clients',
    'used_memory_rss': 'used_memory_rss',
    'client_longest_output_list': 'client_longest_output',
    'aof_rewrite_scheduled': 'aof_rewrite_scheduled'
}

ops_rows = []
for key, pat in patterns.items():
    sel = matched[matched['kpi_name'].str.contains(pat, case=False, na=False)]
    if not sel.empty:
        # take the first match's numeric value
        val = float(sel['value'].iloc[0])
    else:
        val = None
    ops_rows.append({'kpi': key, 'value': val})

ops_df = pd.DataFrame(ops_rows)

# Compact outputs (as required): total_matched, kpi_counts, matched_out, ops_df
total_matched, kpi_counts, matched_out, ops_df

```
Out[10]:
```
Summary for Redis02 at exactly 2021-03-04 05:55:00+08:00:

1) Total matched metric rows: 56

2) Top kpi_name examples (most names occur once each; sample top entries):
- OSLinux-CPU_CPU-0_SingleCpuUtil
- OSLinux-CPU_CPU-2_SingleCpuUtil
- OSLinux-CPU_CPU_CPUCpuUtil
- OSLinux-CPU_CPU_CPULoad
- OSLinux-CPU_CPU_CPUSysTime
- OSLinux-CPU_CPU_CPUWio
- OSLinux-CPU_CPU_CPUidleutil
- OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc
- OSLinux-OSLinux_MEMORY_MEMORY_MEMFreeMem
- redis-Redis_6379_redis server
- redis-Redis_6379_Redis  (used_memory)
- redis-Redis_6379_Redis  (used_memory_rss)
- redis-Redis_6379_Redis  (instantaneous_ops_per_sec)
- redis-Redis_6379_Redis  (connected_clients)
- ... (total list truncated; many Redis-specific and OS KPIs; most appear once)

3) Matched rows table (columns: timestamp, cmdb_id, kpi_name, value) — limited sample (sorted by kpi_name). Example rows include:
- 2021-03-04T05:55:00+08:00 — Redis02 — OSLinux-CPU_CPU-0_SingleCpuUtil — 0.4656
- 2021-03-04T05:55:00+08:00 — Redis02 — OSLinux-CPU_CPU-2_SingleCpuUtil — 0.8144
- 2021-03-04T05:55:00+08:00 — Redis02 — OSLinux-CPU_CPU_CPUCpuUtil — 0.3948
- 2021-03-04T05:55:00+08:00 — Redis02 — OSLinux-CPU_CPU_CPULoad — 0.12
- ...
- 2021-03-04T05:55:00+08:00 — Redis02 — OSLinux-CPU_CPU_CPUWio — 0.864400  (notable spike)
- ...
- 2021-03-04T05:55:00+08:00 — Redis02 — redis-Redis_6379_Redis  (used_memory_rss) — 19054592.0
- 2021-03-04T05:55:00+08:00 — Redis02 — redis-Redis_6379_redis server — 1.0

(The full matched table of 56 rows is available in the kernel variable produced by the query.)

4) Requested Redis operational KPIs at that timestamp (each NULL if absent):
- blocked_clients: 0.0
- instantaneous_ops_per_sec: 7.0
- connected_clients: 1398.0
- used_memory_rss: 19054592.0
- client_longest_output_list: 0.0
- aof_rewrite_scheduled: 0.0

Notable points:
- The CPU WIO metric (OSLinux-CPU_CPU_CPUWio) shows a large spike at 05:55:00 (0.864400), far above nearby values — consistent with earlier anomaly detection.
- Connected clients is relatively large (1398) while instantaneous ops/sec is low (7), which may indicate many idle connections or blocked throughput; however blocked_clients is 0.
- Memory RSS ~19,054,592 bytes (≈19 MB) is reported at that time.

If you want, I can print the complete 56-row table here or filter it further (e.g., show only CPU-related rows or only redis_* KPIs).

The original code execution output of IPython Kernel is also provided below for reference:

(56,                                              kpi_name  count
0                     OSLinux-CPU_CPU-0_SingleCpuUtil      1
1                     OSLinux-CPU_CPU-2_SingleCpuUtil      1
2                          OSLinux-CPU_CPU_CPUCpuUtil      1
3                             OSLinux-CPU_CPU_CPULoad      1
4                          OSLinux-CPU_CPU_CPUSysTime      1
..                                                ...    ...
45  redis-Redis_6379_Redis  (total_commands_proces...      1
46  redis-Redis_6379_Redis  (total_connections_rec...      1
47        redis-Redis_6379_Redis  (uptime_in_seconds)      1
48             redis-Redis_6379_Redis  (used_cpu_sys)      1
49    redis-Redis_6379_Redis  (used_cpu_sys_children)      1

[50 rows x 2 columns],                     timestamp  cmdb_id                                          kpi_name         value
0   2021-03-04T05:55:00+08:00  Redis02                   OSLinux-CPU_CPU-0_SingleCpuUtil  4.656000e-01
1   2021-03-04T05:55:00+08:00  Redis02                   OSLinux-CPU_CPU-2_SingleCpuUtil  8.144000e-01
2   2021-03-04T05:55:00+08:00  Redis02                        OSLinux-CPU_CPU_CPUCpuUtil  3.948000e-01
3   2021-03-04T05:55:00+08:00  Redis02                           OSLinux-CPU_CPU_CPULoad  1.200000e-01
4   2021-03-04T05:55:00+08:00  Redis02                        OSLinux-CPU_CPU_CPUSysTime  3.283000e-01
..                        ...      ...                                               ...           ...
51  2021-03-04T05:55:00+08:00  Redis02  redis-Redis_6379_Redis  (used_cpu_user_children)  0.000000e+00
52  2021-03-04T05:55:00+08:00  Redis02             redis-Redis_6379_Redis  (used_memory)  3.374638e+07
53  2021-03-04T05:55:00+08:00  Redis02        redis-Redis_6379_Redis  (used_memory_peak)  9.570678e+07
54  2021-03-04T05:55:00+08:00  Redis02         redis-Redis_6379_Redis  (used_memory_rss)  1.905459e+07
55  2021-03-04T05:55:00+08:00  Redis02                     redis-Redis_6379_redis server  1.000000e+00

[56 rows x 4 columns],                           kpi       value
0             blocked_clients         0.0
1   instantaneous_ops_per_sec         7.0
2           connected_clients      1398.0
3             used_memory_rss  19054592.0
4  client_longest_output_list         0.0
5       aof_rewrite_scheduled         0.0)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```